In [43]:
import numpy as np
import pandas as pd
from pathlib import Path
from sentence_transformers import SentenceTransformer

In [44]:
candidate_sites = pd.read_parquet("../../data/processed/retrieval/candidate_sites.parquet")
training_pairs = pd.read_parquet("../../data/processed/retrieval/training_pairs.parquet")
queries = pd.read_json("../../data/processed/retrieval/query_intents.jsonl", lines=True)

print("Candidates:", len(candidate_sites))
print("Training pairs:", len(training_pairs))
print("Queries:", len(queries))

Candidates: 49950
Training pairs: 12600
Queries: 21


In [45]:
print(candidate_sites.columns)
print(training_pairs.columns)
print(queries.columns)

Index(['RID', 'address', 'primary_zoning_code', 'primary_zoning_class',
       'zoning_band', 'mixed_zoning_flag', 'lot_size_proxy_sqm',
       'lot_size_band', 'heritage_flag', 'heritage_max_significance',
       'bushfire_flag', 'bushfire_risk_level', 'flood_flag',
       'primary_flood_class', 'distance_to_station_m', 'within_800m_catchment',
       'station_distance_band', 'station_access_score',
       'constraint_severity_band', 'top_strategy', 'top_strategy_score',
       'site_summary_text', 'candidate_text_debug', 'candidate_text_clean',
       'single_dwelling_rebuild_score', 'assembly_opportunity_score',
       'granny_flat_score', 'land_bank_hold_score',
       'townhouse_multi_dwelling_score', 'low_rise_apartment_score',
       'dual_occupancy_score'],
      dtype='object')
Index(['query_id', 'strategy', 'query_text', 'candidate_rid', 'candidate_text',
       'candidate_score', 'label', 'pair_type', 'address', 'top_strategy',
       'top_strategy_score', 'primary_zoning_co

In [46]:
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(MODEL_NAME)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [29]:
candidate_sites["candidate_text"] = candidate_sites["candidate_text"].fillna("").astype(str)
queries["text"] = queries["text"].fillna("").astype(str)

candidate_texts = candidate_sites["candidate_text"].tolist()
query_texts = queries["text"].tolist()

print(candidate_texts[:2])
print(query_texts[:2])

['top_strategy single_dwelling_rebuild | strategy_signals dual_occupancy granny_flat single_dwelling_rebuild | zoning_code R2 | zoning_band low_dev | lot_size_band m | station_distance_band over_10km | constraint_severity low | mixed_zoning no | heritage no | flood no | bushfire_risk 0 | within_800m no', 'top_strategy granny_flat | strategy_signals granny_flat | zoning_code R5 | zoning_band low_dev | lot_size_band xl | station_distance_band 2km_5km | constraint_severity high | mixed_zoning no | heritage no | flood no | bushfire_risk 3 | within_800m no']
['strategy single_dwelling_rebuild | zoning preference low_dev or mid_dev | lot size preference s m l | access preference not critical | avoid heritage flood severe bushfire', 'strategy single_dwelling_rebuild | detached residential rebuild | zoning low_dev mid_dev | moderate lot size | low constraint preferred']


In [30]:
candidate_embeddings = model.encode(
    candidate_texts,
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

candidate_embeddings.shape

Batches:   0%|          | 0/391 [00:00<?, ?it/s]

(49950, 384)

In [31]:
query_embeddings = model.encode(
    query_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

query_embeddings.shape

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

(21, 384)

In [32]:
similarity_matrix = query_embeddings @ candidate_embeddings.T
similarity_matrix.shape

(21, 49950)

In [33]:
def retrieve_top_k(query_idx: int, k: int = 10) -> pd.DataFrame:
    sims = similarity_matrix[query_idx]
    top_idx = np.argsort(-sims)[:k]

    result = candidate_sites.iloc[top_idx].copy()
    result["similarity"] = sims[top_idx]
    result["query_id"] = queries.iloc[query_idx]["query_id"]
    result["query_text"] = queries.iloc[query_idx]["text"]
    result["strategy"] = queries.iloc[query_idx]["strategy"]
    return result

In [34]:
q_idx = 0
top10 = retrieve_top_k(q_idx, k=10)

display(
    top10[
        [
            "query_id",
            "strategy",
            "address",
            "primary_zoning_code",
            "lot_size_band",
            "constraint_severity_band",
            "station_distance_band",
            "top_strategy",
            "top_strategy_score",
            "similarity",
            "candidate_text",
        ]
    ]
)

,query_id,strategy,address,primary_zoning_code,lot_size_band,constraint_severity_band,station_distance_band,top_strategy,top_strategy_score,similarity,candidate_text
13258,q001,single_dwelling_rebuild,26 PRYCE PARADE ABERCROMBIE,R1,l,low,5km_10km,single_dwelling_rebuild,75.0,0.827243,top_strategy single_dwelling_rebuild | strateg...
49438,q001,single_dwelling_rebuild,23 CAMPBELL CLOSE LLANARTH,R1,l,low,5km_10km,single_dwelling_rebuild,75.0,0.827243,top_strategy single_dwelling_rebuild | strateg...
43125,q001,single_dwelling_rebuild,NILE STREET RAGLAN,R1,l,low,5km_10km,single_dwelling_rebuild,75.0,0.827243,top_strategy single_dwelling_rebuild | strateg...
11690,q001,single_dwelling_rebuild,70 ABERCROMBIE DRIVE ABERCROMBIE,R1,l,low,5km_10km,single_dwelling_rebuild,75.0,0.827243,top_strategy single_dwelling_rebuild | strateg...
5960,q001,single_dwelling_rebuild,60 BARRENJOEY ROAD ETTALONG BEACH,R1,s,low,2km_5km,single_dwelling_rebuild,66.2,0.824725,top_strategy single_dwelling_rebuild | strateg...
8545,q001,single_dwelling_rebuild,59A BARRENJOEY ROAD ETTALONG BEACH,R1,s,low,2km_5km,single_dwelling_rebuild,66.2,0.824725,top_strategy single_dwelling_rebuild | strateg...
31824,q001,single_dwelling_rebuild,2/11 HUNTS LANE AVOCA BEACH,R1,s,low,2km_5km,single_dwelling_rebuild,76.2,0.824725,top_strategy single_dwelling_rebuild | strateg...
20281,q001,single_dwelling_rebuild,6/7 BEDFORD CRESCENT DULWICH HILL,R1,s,low,2km_5km,single_dwelling_rebuild,66.2,0.824725,top_strategy single_dwelling_rebuild | strateg...
12866,q001,single_dwelling_rebuild,124 SYDNEY ROAD FAIRLIGHT,R1,s,low,2km_5km,single_dwelling_rebuild,66.2,0.824725,top_strategy single_dwelling_rebuild | strateg...
4050,q001,single_dwelling_rebuild,30 BREAM ROAD ETTALONG BEACH,R1,s,low,2km_5km,single_dwelling_rebuild,66.2,0.824725,top_strategy single_dwelling_rebuild | strateg...


In [35]:
for q_idx in range(min(7, len(queries))):
    q = queries.iloc[q_idx]
    print("\n" + "=" * 80)
    print("QUERY:", q["query_id"], "|", q["strategy"])
    print(q["text"])

    topk = retrieve_top_k(q_idx, k=5)
    display(
        topk[
            [
                "address",
                "primary_zoning_code",
                "lot_size_band",
                "constraint_severity_band",
                "station_distance_band",
                "top_strategy",
                "top_strategy_score",
                "similarity",
            ]
        ]
    )


QUERY: q001 | single_dwelling_rebuild
strategy single_dwelling_rebuild | zoning preference low_dev or mid_dev | lot size preference s m l | access preference not critical | avoid heritage flood severe bushfire


,address,primary_zoning_code,lot_size_band,constraint_severity_band,station_distance_band,top_strategy,top_strategy_score,similarity
13258,26 PRYCE PARADE ABERCROMBIE,R1,l,low,5km_10km,single_dwelling_rebuild,75.0,0.827243
49438,23 CAMPBELL CLOSE LLANARTH,R1,l,low,5km_10km,single_dwelling_rebuild,75.0,0.827243
43125,NILE STREET RAGLAN,R1,l,low,5km_10km,single_dwelling_rebuild,75.0,0.827243
11690,70 ABERCROMBIE DRIVE ABERCROMBIE,R1,l,low,5km_10km,single_dwelling_rebuild,75.0,0.827243
5960,60 BARRENJOEY ROAD ETTALONG BEACH,R1,s,low,2km_5km,single_dwelling_rebuild,66.2,0.824725



QUERY: q002 | single_dwelling_rebuild
strategy single_dwelling_rebuild | detached residential rebuild | zoning low_dev mid_dev | moderate lot size | low constraint preferred


,address,primary_zoning_code,lot_size_band,constraint_severity_band,station_distance_band,top_strategy,top_strategy_score,similarity
11690,70 ABERCROMBIE DRIVE ABERCROMBIE,R1,l,low,5km_10km,single_dwelling_rebuild,75.0,0.771414
49438,23 CAMPBELL CLOSE LLANARTH,R1,l,low,5km_10km,single_dwelling_rebuild,75.0,0.771414
13258,26 PRYCE PARADE ABERCROMBIE,R1,l,low,5km_10km,single_dwelling_rebuild,75.0,0.771414
43125,NILE STREET RAGLAN,R1,l,low,5km_10km,single_dwelling_rebuild,75.0,0.771414
27285,79 GREENFIELD CRESCENT ELDERSLIE,R1,l,low,5km_10km,single_dwelling_rebuild,75.0,0.766946



QUERY: q003 | single_dwelling_rebuild
strategy single_dwelling_rebuild | standard residential redevelopment | prefers R1 R2 R5 RU5 style context | avoid strong planning constraints


...

,address,primary_zoning_code,lot_size_band,constraint_severity_band,station_distance_band,top_strategy,top_strategy_score,similarity
28897,7/2 CALABRO AVENUE LIVERPOOL,R4,l,low,2km_5km,assembly_opportunity,78.6,0.678484
26586,17/319-323 BLAXCELL STREET SOUTH GRANVILLE,R4,l,low,2km_5km,assembly_opportunity,78.6,0.678484
36796,131/1 BROADWAY PUNCHBOWL,R4,l,low,2km_5km,assembly_opportunity,78.6,0.678484
44263,5/191-201 RAMSGATE ROAD RAMSGATE BEACH,R4,l,low,2km_5km,assembly_opportunity,78.6,0.678484
44262,17/191-201 RAMSGATE ROAD RAMSGATE BEACH,R4,l,low,2km_5km,assembly_opportunity,78.6,0.678484



QUERY: q006 | assembly_opportunity
strategy assembly_opportunity | land aggregation potential | mixed zoning or larger site | strong station access preferred | not low density


,address,primary_zoning_code,lot_size_band,constraint_severity_band,station_distance_band,top_strategy,top_strategy_score,similarity
26586,17/319-323 BLAXCELL STREET SOUTH GRANVILLE,R4,l,low,2km_5km,assembly_opportunity,78.6,0.761331
36796,131/1 BROADWAY PUNCHBOWL,R4,l,low,2km_5km,assembly_opportunity,78.6,0.761331
28897,7/2 CALABRO AVENUE LIVERPOOL,R4,l,low,2km_5km,assembly_opportunity,78.6,0.761331
44263,5/191-201 RAMSGATE ROAD RAMSGATE BEACH,R4,l,low,2km_5km,assembly_opportunity,78.6,0.761331
44262,17/191-201 RAMSGATE ROAD RAMSGATE BEACH,R4,l,low,2km_5km,assembly_opportunity,78.6,0.761331



QUERY: q007 | granny_flat
strategy granny_flat | zoning preference low_dev | lot size preference m l xl | lower intensity residential use | avoid heritage flood severe bushfire


,address,primary_zoning_code,lot_size_band,constraint_severity_band,station_distance_band,top_strategy,top_strategy_score,similarity
16043,98-104 KENILWORTH CRESCENT CRANEBROOK,R5,xl,moderate,5km_10km,granny_flat,70.5,0.799621
27554,35 APPLE TREE LANE LITTLE HARTLEY,R5,xl,moderate,5km_10km,granny_flat,70.5,0.799621
47142,261 RANGE ROAD WHITTINGHAM,R5,xl,moderate,5km_10km,granny_flat,70.5,0.799621
32498,38 CORAL VALE DRIVE WONGAWILLI,R5,xl,moderate,2km_5km,granny_flat,72.0,0.798857
35067,5 BRAEMAR AVENUE BRAEMAR,R5,xl,moderate,2km_5km,granny_flat,72.0,0.798857


In [36]:
def top_k_strategy_match_rate(query_idx: int, k: int = 10) -> float:
    strategy = queries.iloc[query_idx]["strategy"]
    topk = retrieve_top_k(query_idx, k=k)
    return (topk["top_strategy"] == strategy).mean()

match_rows = []
for i in range(len(queries)):
    match_rows.append(
        {
            "query_id": queries.iloc[i]["query_id"],
            "strategy": queries.iloc[i]["strategy"],
            "top10_match_rate": top_k_strategy_match_rate(i, k=10),
            "top20_match_rate": top_k_strategy_match_rate(i, k=20),
        }
    )

match_df = pd.DataFrame(match_rows)
match_df

,query_id,strategy,top10_match_rate,top20_match_rate
0,q001,single_dwelling_rebuild,1.0,1.0
1,q002,single_dwelling_rebuild,1.0,1.0
2,q003,single_dwelling_rebuild,1.0,1.0
3,q004,assembly_opportunity,1.0,1.0
4,q005,assembly_opportunity,1.0,1.0
5,q006,assembly_opportunity,1.0,1.0
6,q007,granny_flat,1.0,1.0
7,q008,granny_flat,1.0,1.0
8,q009,granny_flat,1.0,1.0
9,q010,land_bank_hold,1.0,1.0


In [37]:
match_df[["top10_match_rate", "top20_match_rate"]].mean()

top10_match_rate    0.857143
top20_match_rate    0.857143
dtype: float64

In [38]:
def score_col_for_strategy(strategy: str) -> str:
    return f"{strategy}_score"

score_check_rows = []
for i in range(len(queries)):
    strategy = queries.iloc[i]["strategy"]
    score_col = score_col_for_strategy(strategy)
    topk = retrieve_top_k(i, k=20)

    score_check_rows.append(
        {
            "query_id": queries.iloc[i]["query_id"],
            "strategy": strategy,
            "top20_mean_score": topk[score_col].mean(),
            "top20_median_score": topk[score_col].median(),
            "top20_high_score_rate": (topk[score_col] >= 70).mean(),
        }
    )

score_check_df = pd.DataFrame(score_check_rows)
score_check_df

,query_id,strategy,top20_mean_score,top20_median_score,top20_high_score_rate
0,q001,single_dwelling_rebuild,70.730,75.0,0.60
1,q002,single_dwelling_rebuild,67.200,67.2,0.40
2,q003,single_dwelling_rebuild,63.120,67.2,0.25
3,q004,assembly_opportunity,75.240,73.6,1.00
4,q005,assembly_opportunity,79.640,78.6,1.00
5,q006,assembly_opportunity,82.550,83.3,1.00
6,q007,granny_flat,71.000,70.5,0.90
7,q008,granny_flat,75.300,76.5,1.00
8,q009,granny_flat,71.500,71.5,1.00
9,q010,land_bank_hold,55.130,55.8,0.00


In [39]:
score_check_df[["top20_mean_score", "top20_median_score", "top20_high_score_rate"]].mean()

top20_mean_score         70.072619
top20_median_score       70.704762
top20_high_score_rate     0.600000
dtype: float64

In [40]:
all_topk = []
for i in range(len(queries)):
    topk = retrieve_top_k(i, k=50).copy()
    all_topk.append(topk)

baseline_retrieval = pd.concat(all_topk, ignore_index=True)

Path("../../data/processed/retrieval").mkdir(parents=True, exist_ok=True)
baseline_retrieval.to_parquet(
    "../../data/processed/retrieval/embedding_baseline_top50.parquet",
    index=False,
)

print(len(baseline_retrieval))
baseline_retrieval.head()

1050


,RID,address,primary_zoning_code,primary_zoning_class,zoning_band,mixed_zoning_flag,lot_size_proxy_sqm,lot_size_band,heritage_flag,heritage_max_significance,...,assembly_opportunity_score,granny_flat_score,land_bank_hold_score,townhouse_multi_dwelling_score,low_rise_apartment_score,dual_occupancy_score,similarity,query_id,query_text,strategy
0,1445164,26 PRYCE PARADE ABERCROMBIE,R1,General Residential,mid_dev,1,2453.468830,l,0,None,...,50.0,75.0,60.0,50.0,27.0,75.0,0.827243,q001,strategy single_dwelling_rebuild | zoning pref...,single_dwelling_rebuild
1,7460943,23 CAMPBELL CLOSE LLANARTH,R1,General Residential,mid_dev,1,2102.974309,l,0,None,...,50.0,75.0,60.0,50.0,27.0,75.0,0.827243,q001,strategy single_dwelling_rebuild | zoning pref...,single_dwelling_rebuild
2,6406659,NILE STREET RAGLAN,R1,General Residential,mid_dev,1,2070.499616,l,0,None,...,50.0,75.0,60.0,50.0,27.0,75.0,0.827243,q001,strategy single_dwelling_rebuild | zoning pref...,single_dwelling_rebuild
3,1273140,70 ABERCROMBIE DRIVE ABERCROMBIE,R1,General Residential,mid_dev,1,2106.102211,l,0,None,...,50.0,75.0,60.0,50.0,27.0,75.0,0.827243,q001,strategy single_dwelling_rebuild | zoning pref...,single_dwelling_rebuild
4,653434,60 BARRENJOEY ROAD ETTALONG BEACH,R1,General Residential,mid_dev,1,545.315516,s,0,None,...,33.6,46.5,33.6,33.0,21.5,46.8,0.824725,q001,strategy single_dwelling_rebuild | zoning pref...,single_dwelling_rebuild


In [41]:
strategy = "low_rise_apartment"
q_idx = queries.index[queries["strategy"] == strategy][0]

topk = retrieve_top_k(q_idx, k=20)
display(
    topk[
        [
            "address",
            "primary_zoning_code",
            "lot_size_proxy_sqm",
            "within_800m_catchment",
            "heritage_flag",
            "bushfire_risk_level",
            "flood_flag",
            "top_strategy",
            "top_strategy_score",
            "low_rise_apartment_score",
            "similarity",
            "candidate_text",
        ]
    ]
)

,address,primary_zoning_code,lot_size_proxy_sqm,within_800m_catchment,heritage_flag,bushfire_risk_level,flood_flag,top_strategy,top_strategy_score,low_rise_apartment_score,similarity,candidate_text
28513,121/212-216 MONA VALE ROAD ST IVES,R4,10748.238686,0,0,0,0,low_rise_apartment,79.5,79.5,0.831332,top_strategy low_rise_apartment | strategy_sig...
32391,44/3 SHORTLAND STREET TELOPEA,R4,8145.630348,0,0,0,0,low_rise_apartment,79.5,79.5,0.831332,top_strategy low_rise_apartment | strategy_sig...
37210,508/314 CANTERBURY ROAD CANTERBURY,R4,8018.923792,0,0,0,0,low_rise_apartment,79.5,79.5,0.831332,top_strategy low_rise_apartment | strategy_sig...
34654,11D/2B MOWBRAY STREET SYLVANIA,R4,7005.207716,0,0,0,0,low_rise_apartment,79.5,79.5,0.831332,top_strategy low_rise_apartment | strategy_sig...
48064,16/1-3 STURT PLACE ST IVES,R4,5276.625055,0,0,0,0,low_rise_apartment,79.5,79.5,0.831332,top_strategy low_rise_apartment | strategy_sig...
4091,9 BENDIGO PLACE CARTWRIGHT,R4,6601.715756,0,0,0,0,low_rise_apartment,79.5,79.5,0.831332,top_strategy low_rise_apartment | strategy_sig...
4092,2/12 BENDIGO PLACE CARTWRIGHT,R4,6601.715756,0,0,0,0,low_rise_apartment,79.5,79.5,0.831332,top_strategy low_rise_apartment | strategy_sig...
32390,40/3 SHORTLAND STREET TELOPEA,R4,8145.630348,0,0,0,0,low_rise_apartment,79.5,79.5,0.831332,top_strategy low_rise_apartment | strategy_sig...
5651,29/12 WOODWARD CRESCENT MILLER,R4,19337.277870,0,0,0,0,low_rise_apartment,79.5,79.5,0.831332,top_strategy low_rise_apartment | strategy_sig...
47682,210/810 CANTERBURY ROAD ROSELANDS,R4,9714.210247,0,0,0,0,low_rise_apartment,79.5,79.5,0.831332,top_strategy low_rise_apartment | strategy_sig...


In [42]:
print(match_df)
print(score_check_df)
print(match_df[["top10_match_rate", "top20_match_rate"]].mean())
print(score_check_df[["top20_mean_score", "top20_median_score", "top20_high_score_rate"]].mean())

   query_id                  strategy  top10_match_rate  top20_match_rate
0      q001   single_dwelling_rebuild               1.0               1.0
1      q002   single_dwelling_rebuild               1.0               1.0
2      q003   single_dwelling_rebuild               1.0               1.0
3      q004      assembly_opportunity               1.0               1.0
4      q005      assembly_opportunity               1.0               1.0
5      q006      assembly_opportunity               1.0               1.0
6      q007               granny_flat               1.0               1.0
7      q008               granny_flat               1.0               1.0
8      q009               granny_flat               1.0               1.0
9      q010            land_bank_hold               1.0               1.0
10     q011            land_bank_hold               1.0               1.0
11     q012            land_bank_hold               1.0               1.0
12     q013  townhouse_multi_dwelling 